# Team agent benchmark · one Golden set, interchangeable AgentWrappers

Every team supplies one Python class implementing a small **AgentWrapper**
lifecycle: `start()`, `answer(question)`, `close()`. The notebook creates
each team's wrapper, starts it **once**, asks all uncached Golden questions,
then closes it, restarting only when a question fails. Initialization and
connection reuse belong to the wrapper, not DeepEval.

All agents receive **the same questions only**; DeepEval scores saved answers
and optional actually retrieved passages against fixed reference answers.
No DeepEval classes or tracers are required inside team wrappers.

## 1 · Configure participants and judge

Each team exposes an importable Python **wrapper class** (not a function).
Its constructor may accept optional `kwargs`. The notebook owns lifecycle
execution and requires an isolated conversation for each Golden.

The built-in `cli-agent` wrapper is an optional implementation example.
Other teams only need to install their modules. Increment `revision` when
agent code/model/config changes to invalidate previously cached answers.
Set `MAX_CASES=None` after the three-question smoke test.

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import importlib
import inspect
import json
import os
import re
import tempfile
import tomllib
from dataclasses import asdict, is_dataclass
from collections.abc import Mapping
from typing import Protocol

WORKSPACE = Path.cwd().resolve()   # workspace containing rag-goldens/
GOLDENS_FILE = WORKSPACE / "rag-goldens" / "deepeval-goldens.json"
OUTPUT_DIR = WORKSPACE / "rag-goldens" / "evaluation"

AGENTS = {
    "cli-agent": {
        "entrypoint": "builtin:cli-agent",
        "revision": "sdk-baseline-v1",
        "system_prompt": None,
        "style_criteria": None,
    },
    # "team-b": {
    #     "entrypoint": "team_b.benchmark:TeamBWrapper",
    #     "revision": "commit-a1b2c3d",
    #     "kwargs": {"endpoint": "http://localhost:8080"},
    #     "style_criteria": None,
    # },
}

# Used only by the optional builtin:cli-agent example
CONFIG_FILE = None
ADMIN_CONFIG_FILE = None
APPROVE_TOOLS = ()

MAX_CASES = 3
TIMEOUT_SECONDS = 600
FRESH_RESPONSES = False
CACHE_EPOCH = ""  # update after changing corpus, indexes or external services

# Same independently chosen judge for every agent
JUDGE_PROVIDER = "openrouter"  # openrouter | openai | ollama
JUDGE_MODEL = "openai/gpt-4.1"
JUDGE_BASE_URL = None
THRESHOLD = 0.7
JUDGE_MAX_CONCURRENT = 4
FRESH_EVALUATION = False
# Tool matching is optional: Goldens need expected_tools to enable it.
# With False, ToolCorrectness compares tool identities, not input arguments.
TOOL_COMPARE_ARGUMENTS = False

## 2 · Load one immutable DeepEval dataset and an independent judge

No agent code is needed to load the dataset. DeepEval owns only the
**evaluation stage**: the agents never see `expected_output`.

In [ ]:
from deepeval.dataset import EvaluationDataset
from deepeval.models import OpenAIModel, OpenRouterModel, OllamaModel

dataset = EvaluationDataset()
dataset.add_goldens_from_json_file(str(GOLDENS_FILE))
if MAX_CASES is not None:
    dataset = EvaluationDataset(goldens=dataset.goldens[:MAX_CASES])
if not dataset.goldens:
    raise ValueError("The shared Golden set is empty")
if len({g.additional_metadata["golden_id"] for g in dataset.goldens}) != len(dataset.goldens):
    raise ValueError("Duplicate golden_id in the dataset")

if not AGENTS or any(not re.fullmatch(r"[a-z][a-z0-9_-]{0,40}", name) for name in AGENTS):
    raise ValueError("Agent IDs must be unique lowercase identifiers")
for name, spec in AGENTS.items():
    if not isinstance(spec, dict):
        raise TypeError(f"{name}: agent specification must be a dictionary")
    if not isinstance(spec.get("entrypoint"), str) or not spec["entrypoint"].strip():
        raise ValueError(f"{name}: entrypoint is required")
    if not isinstance(spec.get("revision"), str) or not spec["revision"].strip():
        raise ValueError(f"{name}: nonempty revision is required for cache correctness")
    style = spec.get("style_criteria")
    if style is not None and (not isinstance(style, str) or not style.strip()):
        raise ValueError(f"{name}: style_criteria must be text or None")
    kwargs = spec.get("kwargs", {})
    if not isinstance(kwargs, dict) or not all(
        isinstance(k, str) for k in kwargs
    ):
        raise ValueError(f"{name}: kwargs must be a dictionary with string keys")
    try:
        json.dumps(kwargs, sort_keys=True)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name}: kwargs must be JSON-serializable for caching") from exc

if JUDGE_PROVIDER == "openrouter":
    if JUDGE_BASE_URL:
        raise ValueError("OpenRouter uses its fixed endpoint")
    judge = OpenRouterModel(model=JUDGE_MODEL, api_key=os.environ["OPENROUTER_API_KEY"])
elif JUDGE_PROVIDER == "openai":
    judge = OpenAIModel(
        model=JUDGE_MODEL, api_key=os.environ["OPENAI_API_KEY"],
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
elif JUDGE_PROVIDER == "ollama":
    judge = OllamaModel(
        model=JUDGE_MODEL,
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
else:
    raise ValueError(f"Unsupported judge provider: {JUDGE_PROVIDER}")
print(f"{len(dataset.goldens)} identical Goldens | {len(AGENTS)} agents | judge={JUDGE_MODEL}")

## 3 · AgentWrapper interface and factory

Every team implements three asynchronous methods: `start()`, `answer(question)`
and `close()`. This `AgentWrapper` is a structural Python `Protocol`;
there is no required inheritance or DeepEval dependency.

`answer()` may return a string or a mapping:

```python
{
    "answer": "The result.",
    "retrieval_context": ["actual retrieved passage"],  # optional
    "tools_called": [                                    # optional
        {"name": "search", "input_parameters": {"query": "topic"}}
    ]
}
```

`tools_called=None` (or absent) means **not reported**, whereas `[]`
means the wrapper observed **zero tool calls**. Each entry must contain
`name`; optional `input_parameters`, `call_id`, `status` and
`duration_ms` are preserved. Tool arguments may contain secrets and
are cached locally. Only provide them if approved for the benchmark.

The built-in `CliAgentWrapper` starts the SDK once and calls `agent.reset()`
before each Golden. The SDK currently returns `result.tool_calls` with
names, statuses, call IDs and durations, **not input parameters**.
The benchmark does not invent unavailable argument data.

`create_wrapper(spec)` imports each team's class using `module:Class`.
Constructor `kwargs` are configured under `AGENTS`. Every wrapper must
ensure a fresh conversation for each Golden, even when initialization and
connection resources are reused.

In [ ]:
class AgentWrapper(Protocol):
    """Small lifecycle expected from all benchmark participants.

    answer() must isolate each Golden question from previous conversations.
    """
    async def start(self) -> None: ...
    async def answer(self, question: str) -> str | Mapping: ...
    async def close(self) -> None: ...


def cli_agent_retrieval_context(trace):
    """Use only complete host-verified SDK-selected passages."""
    if is_dataclass(trace):
        trace = asdict(trace)
    if not isinstance(trace, dict) or trace.get("truncated") is True:
        return None
    if trace.get("content_included") is not True:
        return None
    selected = trace.get("selected")
    if not isinstance(selected, (list, tuple)) or not selected:
        return None
    passages = [item.get("content") if isinstance(item, dict) else None for item in selected]
    if any(not isinstance(part, str) or not part.strip() for part in passages):
        return None
    return passages


class CliAgentWrapper:
    """SDK example: initialize once; reset conversation before every question."""
    def __init__(self, *, system_prompt=None):
        self.system_prompt = system_prompt
        self.agent = None

    async def start(self):
        from cli_agent.sdk import AgentBuilder
        from cli_agent.settings.config import ensure_default_config_file

        config = Path(CONFIG_FILE) if CONFIG_FILE else ensure_default_config_file()
        admin = Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE else None
        builder = AgentBuilder().with_admin_config(path=admin)
        builder.with_project_config(
            path=config, workspace=WORKSPACE,
            workspace_access="none", response_format="text",
        )
        if APPROVE_TOOLS:
            allowed = frozenset(APPROVE_TOOLS)
            async def approve(name, arguments):
                return name in allowed
            builder.with_approval_callback(approve)
        if self.system_prompt is not None:
            if not isinstance(self.system_prompt, str) or not self.system_prompt.strip():
                raise ValueError("system_prompt must be nonempty text")
            builder.with_system_prompt(self.system_prompt)
        agent = builder.with_knowledge_trace(include_content=True).build()
        self.agent = agent
        await agent.start()

    async def answer(self, question):
        if self.agent is None:
            raise RuntimeError("Wrapper not started")
        # Clear session history, evidences, approvals and tool state
        # while retaining model / MCP connections between questions.
        await self.agent.reset()
        result = await self.agent.ask(question)
        return {
            "answer": result.answer,
            "retrieval_context": cli_agent_retrieval_context(result.knowledge_trace),
            # SDK intentionally redacts input parameters and tool output.
            # Preserve only the safe structured metadata it actually reports.
            "tools_called": [
                {
                    "name": call.name,
                    "call_id": call.call_id,
                    "status": call.status,
                    "duration_ms": call.duration_ms,
                }
                for call in result.tool_calls
            ],
        }

    async def close(self):
        agent, self.agent = self.agent, None
        if agent is not None:
            await agent.close()


def create_wrapper(spec) -> AgentWrapper:
    """Construct one wrapper; the execution loop owns start/close lifecycle."""
    entrypoint = spec["entrypoint"]
    if entrypoint == "builtin:cli-agent":
        if spec.get("kwargs"):
            raise ValueError("Builtin wrapper uses system_prompt, not kwargs")
        wrapper = CliAgentWrapper(system_prompt=spec.get("system_prompt"))
    else:
        if entrypoint.count(":") != 1:
            raise ValueError("entrypoint must be 'python.module:WrapperClass'")
        module_name, class_name = entrypoint.split(":", 1)
        if not module_name or not all(part.isidentifier() for part in module_name.split(".")):
            raise ValueError(f"Invalid wrapper module: {entrypoint!r}")
        if not class_name.isidentifier():
            raise ValueError(f"Invalid wrapper class: {entrypoint!r}")
        factory = getattr(importlib.import_module(module_name), class_name)
        if not inspect.isclass(factory):
            raise TypeError(f"{entrypoint}: entrypoint must refer to a wrapper class")
        wrapper = factory(**spec.get("kwargs", {}))
    for method in ("start", "answer", "close"):
        if not callable(getattr(wrapper, method, None)):
            raise TypeError(f"{entrypoint}: missing {method}()")
    return wrapper


def normalize_tool_calls(value):
    """None means no telemetry; [] means telemetry confirms zero calls.

    input_parameters is optional and may be sensitive. The built-in SDK
    never exposes it. External teams may opt into sending it, in which case
    it is stored in the private response cache.
    """
    if value is None:
        return None
    if not isinstance(value, (list, tuple)):
        raise ValueError("tools_called must be a list or None")
    normalized = []
    for tool in value:
        if not isinstance(tool, Mapping):
            raise ValueError("each tools_called entry must be a mapping")
        name = tool.get("name")
        if not isinstance(name, str) or not name.strip():
            raise ValueError("tools_called entries require a nonempty name")
        entry = {"name": name}
        if "input_parameters" in tool:
            params = tool["input_parameters"]
            if params is not None and not isinstance(params, dict):
                raise ValueError("input_parameters must be a dictionary or None")
            if params is not None:
                entry["input_parameters"] = params
        for key in ("call_id", "status"):
            if key in tool and tool[key] is not None:
                if not isinstance(tool[key], str):
                    raise ValueError(f"tools_called.{key} must be a string")
                entry[key] = tool[key]
        if "duration_ms" in tool and tool["duration_ms"] is not None:
            duration = tool["duration_ms"]
            if (isinstance(duration, bool) or not isinstance(duration, (int, float))
                    or not 0 <= duration < float("inf")):
                raise ValueError("tools_called.duration_ms must be finite and nonnegative")
            entry["duration_ms"] = duration
        # Validate data before it enters our JSON cache; never stringify
        # unknown objects or silently drop argument values.
        try:
            json.dumps(entry, allow_nan=False)
        except (TypeError, ValueError) as exc:
            raise ValueError("tools_called must be JSON-serializable") from exc
        normalized.append(entry)
    return normalized


def normalize_answer(value):
    """Accepted: str or {answer, retrieval_context?, tools_called?}."""
    if isinstance(value, str):
        answer, context, tools = value, None, None
    elif isinstance(value, Mapping):
        answer = value.get("answer")
        context = value.get("retrieval_context")
        tools = normalize_tool_calls(value.get("tools_called"))
    else:
        raise TypeError("Agent must return a string or an answer mapping")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("Agent returned an empty answer")
    if context is not None:
        if (not isinstance(context, (list, tuple))
                or any(
                    not isinstance(part, str) or not part.strip()
                    for part in context
                )):
            raise ValueError("retrieval_context must be a list of nonempty strings")
        # No passages is a valid retrieval outcome, not an agent failure.
        context = list(context) or None
    return {
        "actual_output": answer,
        "retrieval_context": context,
        "tools_called": tools,
    }

async def invoke_agent(wrapper, question):
    """Timeout and normalize one independent Golden question."""
    result = wrapper.answer(question)
    if not inspect.isawaitable(result):
        raise TypeError("AgentWrapper.answer must be async")
    return normalize_answer(await asyncio.wait_for(result, timeout=TIMEOUT_SECONDS))

## 4 · Lifecycle, execution and per-agent response caching

The notebook calls `create_wrapper()` only if there are uncached questions
for the agent. It then runs `start()` once, `answer(question)` per Golden,
and `close()` in a `finally` block. If one question fails or times out,
the wrapper is **closed and reconstructed** before the next question to
avoid reusing an invalid or contaminated session.

The wrapper itself guarantees that each question uses a **fresh conversation**,
even while expensive model/index/MCP resources remain initialized.
A built-in `cli-agent` wrapper does this via `agent.reset()`.

Answer caching is separate from DeepEval's metric cache. The `revision`,
`kwargs`, system prompt and agent entrypoint are part of its key; external
indexes or weights must be tracked by `CACHE_EPOCH` or refreshed manually.

In [ ]:
def agent_fingerprint(name, spec):
    digest = hashlib.sha256(json.dumps({
        "name": name, "entrypoint": spec["entrypoint"],
        "revision": spec["revision"], "system_prompt": spec.get("system_prompt"),
        "kwargs": spec.get("kwargs", {}),
        # Invalidate caches written before tool telemetry was captured.
        "response_schema_version": 2,
        "workspace": str(WORKSPACE), "epoch": CACHE_EPOCH,
    }, sort_keys=True).encode("utf-8"))
    if spec["entrypoint"] == "builtin:cli-agent":
        # Other teams' adapters do not need cli-agent installed at all.
        from cli_agent.settings.admin_config import default_admin_config_file
        from cli_agent.settings.config import ensure_default_config_file
        config = Path(CONFIG_FILE) if CONFIG_FILE else ensure_default_config_file()
        admin = Path(ADMIN_CONFIG_FILE) if ADMIN_CONFIG_FILE else default_admin_config_file()
        files = [config]
        if admin.is_file():
            files.append(admin)
        if spec.get("system_prompt") is None:
            values = tomllib.loads(Path(config).read_text(encoding="utf-8"))
            prompt_file = values.get("agent", {}).get("system_prompt")
            if prompt_file:
                files.append(WORKSPACE / ".cli-agent" / prompt_file)
        for file in files:
            file = Path(file).expanduser().resolve(strict=True)
            digest.update(str(file).encode())
            digest.update(file.read_bytes())
        digest.update(json.dumps(sorted(APPROVE_TOOLS)).encode())
    return digest.hexdigest()


def response_file(agent_id, spec, golden):
    key = hashlib.sha256(json.dumps([
        agent_fingerprint(agent_id, spec),
        golden.additional_metadata["golden_id"],
        golden.input, golden.expected_output,
    ], ensure_ascii=False).encode("utf-8")).hexdigest()
    return OUTPUT_DIR / "responses" / f"{key}.json"


def read_cached_response(path):
    if not path.is_file() or path.is_symlink() or path.stat().st_size > 16_000_000:
        return None
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(record, dict):
            return normalize_answer({
                "answer": record.get("actual_output"),
                "retrieval_context": record.get("retrieval_context"),
                "tools_called": record.get("tools_called"),
            })
    except (ValueError, TypeError, OSError):
        pass
    return None


def save_response(path, response):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=path.parent,
            prefix=".response-", suffix=".tmp", delete=False,
        ) as stream:
            temporary = Path(stream.name)
            os.chmod(temporary, 0o600)
            json.dump(response, stream, ensure_ascii=False)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


async def close_wrapper(wrapper):
    """Best-effort cleanup; never conceal an answer exception."""
    if wrapper is None:
        return
    try:
        result = wrapper.close()
        if not inspect.isawaitable(result):
            raise TypeError("AgentWrapper.close must be async")
        await asyncio.wait_for(result, timeout=TIMEOUT_SECONDS)
    except Exception as error:
        print(f"Wrapper cleanup failed: {type(error).__name__}")


all_responses = {}
all_failures = {}
for agent_id, spec in AGENTS.items():
    responses, failures = {}, {}
    wrapper = None
    try:
        for golden in dataset.goldens:
            golden_id = golden.additional_metadata["golden_id"]
            checkpoint = response_file(agent_id, spec, golden)
            cached = None if FRESH_RESPONSES else read_cached_response(checkpoint)
            if cached is not None:
                responses[golden_id] = cached
                continue
            try:
                if wrapper is None:
                    wrapper = create_wrapper(spec)
                    started = wrapper.start()
                    if not inspect.isawaitable(started):
                        raise TypeError("AgentWrapper.start must be async")
                    await asyncio.wait_for(started, timeout=TIMEOUT_SECONDS)
                response = await invoke_agent(wrapper, golden.input)
                save_response(checkpoint, response)
                responses[golden_id] = response
            except Exception as error:
                # A failed request may leave model/tool state unusable.
                # Discard the wrapper and create a fresh one for next case.
                failures[golden_id] = type(error).__name__
                await close_wrapper(wrapper)
                wrapper = None
    finally:
        await close_wrapper(wrapper)
        all_responses[agent_id] = responses
        all_failures[agent_id] = failures
    print(f"{agent_id}: {len(responses)}/{len(dataset.goldens)} answered; "
          f"{len(failures)} failed")
    if failures:
        print("  Failed Golden IDs:", ", ".join(sorted(failures)))

## 5 · DeepEval judges answers, optionally retrieval and tool correctness

Every team gets the **same reference answers**, Judge model and shared
factual metrics. Retrieval metrics use only passages the wrapper actually
provides. Tool metrics use actual `tools_called` only when a Golden
**explicitly** specifies `expected_tools`. Our generated Goldens do
not currently specify expected tools, so tool correctness is **not**
automatically run. This avoids ranking heterogeneous agents by a fixed
tool implementation.

DeepEval's `ToolCorrectnessMetric` compares tool names by default;
`TOOL_COMPARE_ARGUMENTS=True` enables comparison of input parameters
when both actual and expected tool arguments are supplied. Tool counts,
statuses and durations are reported independently of Golden expectations.
`ToolUseMetric` instead targets multi-turn conversations and is not run
on these single-question Goldens.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    GEval, AnswerRelevancyMetric, FaithfulnessMetric,
    ContextualRecallMetric, ContextualPrecisionMetric, ToolCorrectnessMetric,
)
from deepeval.test_case import LLMTestCase, SingleTurnParams, ToolCall, ToolCallParams
from deepeval.evaluate import AsyncConfig, CacheConfig, DisplayConfig

evaluation_options = {
    "cache_config": CacheConfig(use_cache=not FRESH_EVALUATION, write_cache=True),
    "display_config": DisplayConfig(
        results_folder=str(OUTPUT_DIR / "deepeval"),
        inspect_after_run=False,
    ),
}

evaluation_results = {}
for agent_id, spec in AGENTS.items():
    responses = all_responses[agent_id]
    answer_cases, retrieval_cases, tool_cases = [], [], []
    for golden in dataset.goldens:
        response = responses.get(golden.additional_metadata["golden_id"])
        if response is None:
            continue  # failures are counted separately in the summary
        base = {
            "input": golden.input,
            "expected_output": golden.expected_output,
            "actual_output": response["actual_output"],
        }
        answer_cases.append(LLMTestCase(**base))
        if response["retrieval_context"]:
            retrieval_cases.append(LLMTestCase(
                **base, retrieval_context=response["retrieval_context"],
            ))
        # Tool correctness needs a known expected tool set. Never infer one
        # from the answer or from another team's implementation.
        if response["tools_called"] is not None and golden.expected_tools is not None:
            tool_cases.append(LLMTestCase(
                **base,
                tools_called=[
                    ToolCall(
                        name=call["name"],
                        **({"input_parameters": call["input_parameters"]}
                           if "input_parameters" in call else {}),
                    ) for call in response["tools_called"]
                ],
                expected_tools=golden.expected_tools,
            ))

    correctness = GEval(
        name="Answer correctness",
        criteria=(
            "Judge factual correctness and completeness against the fixed reference. "
            "Accept paraphrases, short or detailed answers equally when all "
            "necessary facts and caveats are preserved. Penalize invented facts."
        ),
        evaluation_params=[
            SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
            SingleTurnParams.EXPECTED_OUTPUT,
        ],
        model=judge, threshold=THRESHOLD,
    )
    metrics = [
        correctness,
        AnswerRelevancyMetric(model=judge, threshold=THRESHOLD),
    ]
    if spec.get("style_criteria"):
        metrics.append(GEval(
            name="Style compliance",
            criteria=(
                "Score only response style, not factual correctness. Goal: "
                + spec["style_criteria"]
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
            ],
            model=judge, threshold=THRESHOLD,
        ))

    answer_results = evaluate(
        test_cases=answer_cases, metrics=metrics,
        identifier=f"rag-{agent_id}-answer",
        async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
        **evaluation_options,
    ) if answer_cases else None

    retrieval_results = None
    if retrieval_cases:
        uncertainty = GEval(
            name="Uncertainty calibration",
            criteria=(
                "Using the actual retrieved passages, assess justified "
                "confidence or limitations in the answer. Penalize unsupported "
                "claims and unnecessary refusal when retrieved evidence supports "
                "an answer. The reference is the factual target, NOT proof that "
                "the agent retrieved it."
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT,
            ],
            model=judge, threshold=THRESHOLD,
        )
        retrieval_results = evaluate(
            test_cases=retrieval_cases,
            metrics=[
                FaithfulnessMetric(model=judge, threshold=THRESHOLD),
                ContextualRecallMetric(model=judge, threshold=THRESHOLD),
                ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
                uncertainty,
            ],
            identifier=f"rag-{agent_id}-retrieval",
            async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
            **evaluation_options,
        )
    tool_results = None
    if tool_cases:
        tool_results = evaluate(
            test_cases=tool_cases,
            metrics=[
                ToolCorrectnessMetric(
                    model=judge, threshold=THRESHOLD,
                    evaluation_params=(
                        [ToolCallParams.INPUT_PARAMETERS]
                        if TOOL_COMPARE_ARGUMENTS else []
                    ),
                ),
            ],
            identifier=f"rag-{agent_id}-tools",
            async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
            **evaluation_options,
        )
    evaluation_results[agent_id] = {
        "answer": answer_results,
        "retrieval": retrieval_results,
        "tools": tool_results,
        "successful_cases": len(answer_cases),
        "retrieval_cases": len(retrieval_cases),
        "tool_cases": len(tool_cases),
    }

## 6 · Compare scores, tool activity and coverage

The comparison table shows all agent failures and the number of Goldens
actually answered, so that incomplete runs cannot appear superior merely
because they skipped hard questions.

Tool metadata is **optional**: `tool_telemetry_cases` distinguishes teams
that report actual tool usage from teams that do not; `total_tool_calls`,
error statuses and argument coverage only describe reported records.
Tool correctness scores exist only for Goldens explicitly defining
expected tools and for agents reporting tool telemetry. Technical tool names
may differ across agent implementations; compare these scores only where
the Golden's tool expectations make sense for every participant.

In [ ]:
from collections import defaultdict
import pandas as pd
from IPython.display import display

summary_rows = []
for agent_id, results in evaluation_results.items():
    scores = defaultdict(list)
    judge_errors = defaultdict(int)
    for section in ("answer", "retrieval", "tools"):
        run = results[section]
        for testcase in (getattr(run, "test_results", []) if run else []):
            for metric in testcase.metrics_data or []:
                name = metric.name.removesuffix(" [GEval]")
                if metric.error or metric.score is None:
                    judge_errors[name] += 1
                else:
                    scores[name].append(float(metric.score))
    row = {
        "agent": agent_id,
        "goldens": len(dataset.goldens),
        "answered": results["successful_cases"],
        "agent_failures": len(all_failures[agent_id]),
        "retrieval_cases": results["retrieval_cases"],
        "tool_correctness_cases": results["tool_cases"],
        "tool_telemetry_cases": sum(
            response["tools_called"] is not None
            for response in all_responses[agent_id].values()
        ),
        "total_tool_calls": sum(
            len(response["tools_called"] or [])
            for response in all_responses[agent_id].values()
        ),
        "tool_error_calls": sum(
            call.get("status") in {"error", "failed", "interrupted"}
            for response in all_responses[agent_id].values()
            for call in (response["tools_called"] or [])
        ),
        "tool_calls_with_arguments": sum(
            "input_parameters" in call
            for response in all_responses[agent_id].values()
            for call in (response["tools_called"] or [])
        ),
    }
    for name in (
        "Answer correctness", "Answer Relevancy",
        "Faithfulness", "Contextual Recall", "Contextual Precision",
        "Uncertainty calibration", "Style compliance",
        "Tool Correctness",
    ):
        values = scores.get(name, [])
        row[name] = round(sum(values) / len(values), 3) if values else None
        row[name + " judged"] = len(values)
        row[name + " judge_errors"] = judge_errors[name]
    summary_rows.append(row)

display(pd.DataFrame(summary_rows))
print("Reports:", OUTPUT_DIR / "deepeval")

### Interpretation and reproducibility

- Everyone uses the same Golden questions, unchanged references, judge
  model and shared factual rubrics. Every wrapper isolates question histories
  and is started once per uninterrupted set of uncached questions.
- For controlled comparisons, use a fixed corpus/index snapshot, model
  settings and adapter `revision`, and set `MAX_CASES=None`.
- A result without `retrieval_context` can still be judged for correctness,
  but not for actual retrieval grounding. Never substitute Golden quotations.
- Tool telemetry is optional. `cli-agent` supplies names, statuses and
  durations but deliberately omits raw arguments and output. Tool correctness
  requires explicit `expected_tools` in the Golden; our generated dataset
  does not currently define them.
- Changing another team's implementation, prompt, model or external index
  requires an updated `revision` / `CACHE_EPOCH` or
  `FRESH_RESPONSES=True`. Judge cache is independent.
- Stored responses and DeepEval reports may contain sensitive data. Keep
  `rag-goldens/evaluation/` out of version control.